## Student Attendance Analytics with Pandas

In [2]:

# Prevent OpenBLAS memory fragmentation / kernel crashes on Windows
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

import pandas as pd
import numpy as np
import pandas as pd
import numpy as np

#### Section 0: Data Loading

In [3]:
df = pd.read_excel("aids_attendance.xlsx")
df

,Roll No,Student Name,Subject,Total Classes,Present,Absent,Excused,Attendance %,Status,Classes Needed (75%),Last Present Date,Consecutive Absences
0,4MW23AD50,SRAJAN S,Data Science & GenAI - 3nd Yr- AI&DS,11,0,11,0,0%,Critical,33 more,Never,11
1,4MW24AD001,ADITHYA S POOJARY,Data Science & GenAI - 3nd Yr- AI&DS,11,7,4,0,64%,Critical,5 more,2026-10-07,0
2,4MW24AD002,ANKITHA PRASANNA NAYAK,Data Science & GenAI - 3nd Yr- AI&DS,11,11,0,0,100%,Safe,✓ Met,2026-10-07,0
3,4MW24AD003,ANUSHA KOTEGAR,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,91%,Safe,✓ Met,2026-10-07,0
4,4MW24AD004,ANVITA RAMESH NAIK,Data Science & GenAI - 3nd Yr- AI&DS,11,9,2,0,82%,Warning,✓ Met,2026-10-07,0
...,...,...,...,...,...,...,...,...,...,...,...,...
57,4MW24AD059,ZEHA AYESHA,Data Science & GenAI - 3nd Yr- AI&DS,10,9,1,0,90%,Safe,✓ Met,2026-10-07,0
58,4MW24AD060,AMITH G PALANDYE,Data Science & GenAI - 3nd Yr- AI&DS,10,0,10,0,0%,Critical,30 more,Never,10
59,4MW24AD061,AMOGH BHASKAR NAIK,Data Science & GenAI - 3nd Yr- AI&DS,10,5,5,0,50%,Critical,10 more,2026-10-07,0
60,4MW24AD062,CHANDISH,Data Science & GenAI - 3nd Yr- AI&DS,10,6,4,0,60%,Critical,6 more,2026-10-05,1


#### Section 1: Data Inspection

In [4]:
"""Q1. Display the first 10 rows, the shape of the DataFrame, and the data types of each column."""
display(df.head(10))
print("Shape of DataFrame:", df.shape)
print("\nData Types:")
print(df.dtypes)


,Roll No,Student Name,Subject,Total Classes,Present,Absent,Excused,Attendance %,Status,Classes Needed (75%),Last Present Date,Consecutive Absences
0,4MW23AD50,SRAJAN S,Data Science & GenAI - 3nd Yr- AI&DS,11,0,11,0,0%,Critical,33 more,Never,11
1,4MW24AD001,ADITHYA S POOJARY,Data Science & GenAI - 3nd Yr- AI&DS,11,7,4,0,64%,Critical,5 more,2026-10-07,0
2,4MW24AD002,ANKITHA PRASANNA NAYAK,Data Science & GenAI - 3nd Yr- AI&DS,11,11,0,0,100%,Safe,✓ Met,2026-10-07,0
3,4MW24AD003,ANUSHA KOTEGAR,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,91%,Safe,✓ Met,2026-10-07,0
4,4MW24AD004,ANVITA RAMESH NAIK,Data Science & GenAI - 3nd Yr- AI&DS,11,9,2,0,82%,Warning,✓ Met,2026-10-07,0
5,4MW24AD005,ARPITHA M THANTRY,Data Science & GenAI - 3nd Yr- AI&DS,11,8,3,0,73%,Critical,1 more,2026-10-07,0
6,4MW24AD007,ASNIYA,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,91%,Safe,✓ Met,2026-10-07,0
7,4MW24AD008,B. G PRARTHANA HEBBAR,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,91%,Safe,✓ Met,2026-10-07,0
8,4MW24AD009,BHARATHRAJ A NAYAK,Data Science & GenAI - 3nd Yr- AI&DS,11,4,7,0,36%,Critical,17 more,2026-10-05,1
9,4MW24AD010,BHEEMA KATTIMANI,Data Science & GenAI - 3nd Yr- AI&DS,11,8,3,0,73%,Critical,1 more,2026-10-07,0


Shape of DataFrame: (62, 12)

Data Types:
Roll No                   str
Student Name              str
Subject                   str
Total Classes           int64
Present                 int64
Absent                  int64
Excused                 int64
Attendance %              str
Status                    str
Classes Needed (75%)      str
Last Present Date         str
Consecutive Absences    int64
dtype: object


In [5]:
"""Q2. Check for missing values in each column. Identify which columns have null values (especially Last Present Date with "Never") and report the count. """
print("Missing values per column:")
print(df.isnull().sum())

never_count = (df['Last Present Date'] == 'Never').sum()
print(f"\nNumber of students with 'Never' as Last Present Date: {never_count}")

Missing values per column:
Roll No                 0
Student Name            0
Subject                 0
Total Classes           0
Present                 0
Absent                  0
Excused                 0
Attendance %            0
Status                  0
Classes Needed (75%)    0
Last Present Date       0
Consecutive Absences    0
dtype: int64

Number of students with 'Never' as Last Present Date: 5


In [6]:
"""Q3. Rename the columns to snake_case (e.g., Roll No → roll_no, Student Name → student_name, Attendance % → attendance_pct). Display the updated column list."""
df.columns = (
    df.columns.str.lower()
    .str.replace(' ', '_')
    .str.replace('%', 'pct')
    .str.replace('[()]', '', regex=True)
)
print("Updated columns:", df.columns.tolist())

Updated columns: ['roll_no', 'student_name', 'subject', 'total_classes', 'present', 'absent', 'excused', 'attendance_pct', 'status', 'classes_needed_75pct', 'last_present_date', 'consecutive_absences']


In [7]:
"""Q4. Convert Attendance % from string format (e.g., "64%") to a numeric float column (0.64 or 64.0). Also convert Last Present Date to datetime, handling "Never" as NaT."""
# Convert attendance percentage string (e.g., "64%") to float (0.64)
df['attendance_pct'] = df['attendance_pct'].str.rstrip('%').astype(float) / 100

# Convert Last Present Date to datetime, turning "Never" into NaT
df['last_present_date'] = pd.to_datetime(df['last_present_date'], errors='coerce')

C:\Users\DELL\AppData\Local\Temp\ipykernel_9848\3942730936.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['last_present_date'] = pd.to_datetime(df['last_present_date'], errors='coerce')


#### Section 2: Filtering & Selection

In [8]:
"""Q5. Filter and display all students whose Status is "Critical". Sort them by Attendance % in ascending order."""
critical_students = df[df['status'] == 'Critical'].sort_values(by='attendance_pct', ascending=True)
display(critical_students)

,roll_no,student_name,subject,total_classes,present,absent,excused,attendance_pct,status,classes_needed_75pct,last_present_date,consecutive_absences
0,4MW23AD50,SRAJAN S,Data Science & GenAI - 3nd Yr- AI&DS,11,0,11,0,0.00,Critical,33 more,NaT,11
58,4MW24AD060,AMITH G PALANDYE,Data Science & GenAI - 3nd Yr- AI&DS,10,0,10,0,0.00,Critical,30 more,NaT,10
52,4MW24AD054,UJWAL. K,Data Science & GenAI - 3nd Yr- AI&DS,10,0,10,0,0.00,Critical,30 more,NaT,10
45,4MW24AD046,SHARANU PATIL,Data Science & GenAI - 3nd Yr- AI&DS,10,0,10,0,0.00,Critical,30 more,NaT,10
36,4MW24AD037,RISHAN FRANCIS MENEZES,Data Science & GenAI - 3nd Yr- AI&DS,10,0,10,0,0.00,Critical,30 more,NaT,10
61,4MW25AD400,SAMPATH S HEBBAR,Data Science & GenAI - 3nd Yr- AI&DS,10,1,9,0,0.10,Critical,26 more,2026-09-02,8
26,4MW24AD027,PRAJWAL,Data Science & GenAI - 3nd Yr- AI&DS,11,3,8,0,0.27,Critical,21 more,2026-10-05,1
30,4MW24AD031,PRAVEEN KUMAR N,Data Science & GenAI - 3nd Yr- AI&DS,10,3,7,0,0.30,Critical,18 more,2026-10-07,0
8,4MW24AD009,BHARATHRAJ A NAYAK,Data Science & GenAI - 3nd Yr- AI&DS,11,4,7,0,0.36,Critical,17 more,2026-10-05,1
31,4MW24AD032,PREETH SHETTY,Data Science & GenAI - 3nd Yr- AI&DS,10,5,5,0,0.50,Critical,10 more,2026-10-05,1


In [9]:
"""Q6. Extract all students who have 0 present days (fully absent). List their roll numbers, names, and consecutive absences."""
fully_absent = df[df['present'] == 0][['roll_no', 'student_name', 'consecutive_absences']]
display(fully_absent)

,roll_no,student_name,consecutive_absences
0,4MW23AD50,SRAJAN S,11
36,4MW24AD037,RISHAN FRANCIS MENEZES,10
45,4MW24AD046,SHARANU PATIL,10
52,4MW24AD054,UJWAL. K,10
58,4MW24AD060,AMITH G PALANDYE,10


In [10]:
"""Q7. Using .loc and .iloc, display:
(a) All "Safe" students using .loc
(b) The first 15 rows and columns 2-6 using .iloc """
# (a) All Safe students using .loc
safe_students = df.loc[df['status'] == 'Safe']
display(safe_students)

# (b) First 15 rows and columns 2-6 using .iloc
subset_iloc = df.iloc[:15, 2:6]
display(subset_iloc)

,roll_no,student_name,subject,total_classes,present,absent,excused,attendance_pct,status,classes_needed_75pct,last_present_date,consecutive_absences
2,4MW24AD002,ANKITHA PRASANNA NAYAK,Data Science & GenAI - 3nd Yr- AI&DS,11,11,0,0,1.00,Safe,✓ Met,2026-10-07,0
3,4MW24AD003,ANUSHA KOTEGAR,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,0.91,Safe,✓ Met,2026-10-07,0
6,4MW24AD007,ASNIYA,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,0.91,Safe,✓ Met,2026-10-07,0
7,4MW24AD008,B. G PRARTHANA HEBBAR,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,0.91,Safe,✓ Met,2026-10-07,0
14,4MW24AD015,K MANYA RAO,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,0.91,Safe,✓ Met,2026-10-07,0
15,4MW24AD016,KAVYA,Data Science & GenAI - 3nd Yr- AI&DS,11,11,0,0,1.00,Safe,✓ Met,2026-10-07,0
28,4MW24AD029,PRATHEEKSHA NAYAK,Data Science & GenAI - 3nd Yr- AI&DS,11,11,0,0,1.00,Safe,✓ Met,2026-10-07,0
32,4MW24AD033,PRIYADARSHINI,Data Science & GenAI - 3nd Yr- AI&DS,10,10,0,0,1.00,Safe,✓ Met,2026-10-07,0
33,4MW24AD034,RAKSHITHA (ASHOK KUMAR),Data Science & GenAI - 3nd Yr- AI&DS,10,10,0,0,1.00,Safe,✓ Met,2026-10-07,0
35,4MW24AD036,RAMITHA R RAO,Data Science & GenAI - 3nd Yr- AI&DS,10,9,1,0,0.90,Safe,✓ Met,2026-10-07,0


,subject,total_classes,present,absent
0,Data Science & GenAI - 3nd Yr- AI&DS,11,0,11
1,Data Science & GenAI - 3nd Yr- AI&DS,11,7,4
2,Data Science & GenAI - 3nd Yr- AI&DS,11,11,0
3,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1
4,Data Science & GenAI - 3nd Yr- AI&DS,11,9,2
5,Data Science & GenAI - 3nd Yr- AI&DS,11,8,3
6,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1
7,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1
8,Data Science & GenAI - 3nd Yr- AI&DS,11,4,7
9,Data Science & GenAI - 3nd Yr- AI&DS,11,8,3


In [11]:
"""Q8. Find all students whose Consecutive Absences is greater than or equal to 5. Display their name, status, and last present date."""
high_absences = df[df['consecutive_absences'] >= 5][['student_name', 'status', 'last_present_date']]
display(high_absences)

,student_name,status,last_present_date
0,SRAJAN S,Critical,NaT
36,RISHAN FRANCIS MENEZES,Critical,NaT
45,SHARANU PATIL,Critical,NaT
52,UJWAL. K,Critical,NaT
58,AMITH G PALANDYE,Critical,NaT
61,SAMPATH S HEBBAR,Critical,2026-09-02


### Section 3: Grouping & Aggregation 

In [12]:
"""Q9. Group the data by Status and compute: 
a) Count of students 
b) Average attendance %  
c) Maximum consecutive absences"""
status_summary = df.groupby('status').agg(
    student_count=('roll_no', 'count'),
    avg_attendance=('attendance_pct', 'mean'),
    max_consecutive_absences=('consecutive_absences', 'max')
)
display(status_summary)

,student_count,avg_attendance,max_consecutive_absences
status,,,
Critical,32,0.498750,11
Safe,23,0.940870,1
Warning,7,0.814286,1


In [13]:
"""Q10. Create a new column Attendance Category:
≥ 90% → "Excellent"
75-89% → "Good"
60-74% → "Average"
< 60% → "Poor"
Then use groupby() to count students in each category."""
def categorize_attendance(pct):
    if pct >= 0.90:
        return "Excellent"
    elif pct >= 0.75:
        return "Good"
    elif pct >= 0.60:
        return "Average"
    else:
        return "Poor"

df['attendance_category'] = df['attendance_pct'].apply(categorize_attendance)
category_counts = df.groupby('attendance_category').size()
display(category_counts)

attendance_category
Average      19
Excellent    23
Good          7
Poor         13
dtype: int64

In [14]:
"""Q11. Calculate the total number of classes conducted (sum of Total Classes) and the total present/absent counts across all students using .agg()."""
totals = df.agg({
    'total_classes': 'sum',
    'present': 'sum',
    'absent': 'sum'
})
display(totals)

total_classes    650
present          454
absent           196
dtype: int64

In [15]:
"""Q12. Group by Status and use .agg() to find:
a)min, max, and mean of Present
b)sum of Absent"""
status_agg = df.groupby('status').agg({
    'present': ['min', 'max', 'mean'],
    'absent': 'sum'
})
display(status_agg)


present               absent
             min max      mean    sum
status                               
Critical       0   8  5.312500    168
Safe           9  11  9.695652     14
Warning        8   9  8.714286     14

### Section 4: Feature Engineering & Transformation

In [16]:
"""Q13. Create a new column Risk Score using the formula:
Risk Score = (Absent * 0.4) + (Consecutive Absences * 0.6)
Display the top 10 students with the highest risk score."""
df['risk_score'] = (df['absent'] * 0.4) + (df['consecutive_absences'] * 0.6)
top_risk_students = df.sort_values(by='risk_score', ascending=False).head(10)
display(top_risk_students[['student_name', 'absent', 'consecutive_absences', 'risk_score']])

,student_name,absent,consecutive_absences,risk_score
0,SRAJAN S,11,11,11.0
36,RISHAN FRANCIS MENEZES,10,10,10.0
58,AMITH G PALANDYE,10,10,10.0
52,UJWAL. K,10,10,10.0
45,SHARANU PATIL,10,10,10.0
61,SAMPATH S HEBBAR,9,8,8.4
26,PRAJWAL,8,1,3.8
8,BHARATHRAJ A NAYAK,7,1,3.4
30,PRAVEEN KUMAR N,7,0,2.8
37,RITHESH R,5,1,2.6


In [17]:
"""Q14. Add a column Classes Needed (Numeric) by extracting the numeric value from the Classes Needed (75%) column (e.g., "5 more" → 5, "✓ Met" → 0)."""
import pandas as pd
import numpy as np

# 1. Load the dataset (make sure 'attendance.xlsx' or 'aids_attendance.xlsx' is in your working directory)
df = pd.read_excel("aids_attendance.xlsx")

# 2. Rename columns to snake_case (as done in Q3)
df.columns = (
    df.columns.str.lower()
    .str.replace(' ', '_')
    .str.replace('%', 'pct')
    .str.replace('[()]', '', regex=True)
)

# 3. Q14 Solution Code
df['classes_needed_numeric'] = (
    pd.to_numeric(
        df['classes_needed_75pct'].astype(str).str.extract(r'(\d+)', expand=False),
        errors='coerce'
    )
    .fillna(0)
    .astype(int)
)

display(df[['student_name', 'classes_needed_75pct', 'classes_needed_numeric']].head(10))

,student_name,classes_needed_75pct,classes_needed_numeric
0,SRAJAN S,33 more,33
1,ADITHYA S POOJARY,5 more,5
2,ANKITHA PRASANNA NAYAK,✓ Met,0
3,ANUSHA KOTEGAR,✓ Met,0
4,ANVITA RAMESH NAIK,✓ Met,0
5,ARPITHA M THANTRY,1 more,1
6,ASNIYA,✓ Met,0
7,B. G PRARTHANA HEBBAR,✓ Met,0
8,BHARATHRAJ A NAYAK,17 more,17
9,BHEEMA KATTIMANI,1 more,1


In [19]:
"""Q15. Using apply() with a lambda function, create a column Attendance Flag that returns:
a)"Defaulter" if attendance < 75%
b)"Safe" otherwise """
import pandas as pd

# 1. Ensure the DataFrame is loaded and columns are renamed (if not already done in your notebook)
df = pd.read_excel("aids_attendance.xlsx")
df.columns = (
    df.columns.str.lower()
    .str.replace(' ', '_')
    .str.replace('%', 'pct')
    .str.replace('[()]', '', regex=True)
)

# 2. Convert attendance_pct to float (if Q4 hasn't been run yet)
if df['attendance_pct'].dtype == 'object':
    df['attendance_pct'] = df['attendance_pct'].str.rstrip('%').astype(float) / 100

# 3. Q15 Solution Code using apply() and lambda
# Convert attendance_pct to numeric (handles "64%" and 0.64)
df['attendance_pct'] = pd.to_numeric(
    df['attendance_pct'].astype(str).str.replace('%', '', regex=False),
    errors='coerce'
)

# If values are in percent units (e.g., 64 instead of 0.64), convert to fraction
if df['attendance_pct'].max() > 1:
    df['attendance_pct'] = df['attendance_pct'] / 100

df['attendance_flag'] = df['attendance_pct'].apply(
    lambda x: "Defaulter" if pd.notna(x) and x < 0.75 else "Safe"
)

display(df[['student_name', 'attendance_pct', 'attendance_flag']].head(10))

,student_name,attendance_pct,attendance_flag
0,SRAJAN S,0.00,Defaulter
1,ADITHYA S POOJARY,0.64,Defaulter
2,ANKITHA PRASANNA NAYAK,1.00,Safe
3,ANUSHA KOTEGAR,0.91,Safe
4,ANVITA RAMESH NAIK,0.82,Safe
5,ARPITHA M THANTRY,0.73,Defaulter
6,ASNIYA,0.91,Safe
7,B. G PRARTHANA HEBBAR,0.91,Safe
8,BHARATHRAJ A NAYAK,0.36,Defaulter
9,BHEEMA KATTIMANI,0.73,Defaulter


In [20]:
"""Q16. Use pd.cut() to bin students into attendance ranges: [0-50, 50-75, 75-90, 90-100] and label them as ["Critical", "Warning", "Good", "Excellent"]."""
df['attendance_bin'] = pd.cut(
    df['attendance_pct'] * 100,
    bins=[0, 50, 75, 90, 100],
    labels=["Critical", "Warning", "Good", "Excellent"],
    include_lowest=True
)
display(df[['student_name', 'attendance_pct', 'attendance_bin']].head(10))

,student_name,attendance_pct,attendance_bin
0,SRAJAN S,0.00,Critical
1,ADITHYA S POOJARY,0.64,Warning
2,ANKITHA PRASANNA NAYAK,1.00,Excellent
3,ANUSHA KOTEGAR,0.91,Excellent
4,ANVITA RAMESH NAIK,0.82,Good
5,ARPITHA M THANTRY,0.73,Warning
6,ASNIYA,0.91,Excellent
7,B. G PRARTHANA HEBBAR,0.91,Excellent
8,BHARATHRAJ A NAYAK,0.36,Critical
9,BHEEMA KATTIMANI,0.73,Warning


### Section 5: Advanced Analysis 

In [21]:
"""Q17. Using pivot_table(), create a summary showing the count of students by Status and Attendance Category (created in Q10)."""
if 'attendance_category' not in df.columns:
    df['attendance_category'] = pd.cut(
        df['attendance_pct'],
        bins=[-np.inf, 0.60, 0.75, 0.90, np.inf],
        labels=['Poor', 'Average', 'Good', 'Excellent'],
        right=False
    )

pivot_summary = df.pivot_table(
    index='status',
    columns='attendance_category',
    values='roll_no',
    aggfunc='count',
    fill_value=0,
    observed=False
)
display(pivot_summary)

attendance_category,Poor,Average,Good,Excellent
status,,,,
Critical,13,19,0,0
Safe,0,0,0,23
Warning,0,0,7,0


In [22]:
"""Q18. Identify duplicate student names (e.g., "ANANYA", "DISHA") and display all rows where the base name repeats. Use string manipulation to extract the base name."""
df['base_name'] = df['student_name'].str.split().str[0]
duplicate_students = df[df.duplicated(subset=['base_name'], keep=False)]
display(duplicate_students[['student_name', 'base_name', 'roll_no']])

,student_name,base_name,roll_no
18,MOHAMMED SUHAN,MOHAMMED,4MW24AD019
19,MOHAMMED AFREED,MOHAMMED,4MW24AD020
20,MOHAMMED AWAD TONSE,MOHAMMED,4MW24AD021
24,PAVAN BHAT,PAVAN,4MW24AD025
25,PAVAN S POOJARY,PAVAN,4MW24AD026


In [23]:
""""Q19. Merge this dataset with a hypothetical Student_Details DataFrame containing Roll No, Branch, and Semester. Perform an inner join and display the merged result."""
# Creating sample student details dataframe for demonstration
student_details = pd.DataFrame({
    'roll_no': df['roll_no'].unique(),
    'branch': 'AI & DS',
    'semester': '3rd Year'
})

merged_df = pd.merge(df, student_details, on='roll_no', how='inner')
display(merged_df.head(10))

,roll_no,student_name,subject,total_classes,present,absent,excused,attendance_pct,status,classes_needed_75pct,last_present_date,consecutive_absences,attendance_flag,attendance_bin,attendance_category,base_name,branch,semester
0,4MW23AD50,SRAJAN S,Data Science & GenAI - 3nd Yr- AI&DS,11,0,11,0,0.00,Critical,33 more,Never,11,Defaulter,Critical,Poor,SRAJAN,AI & DS,3rd Year
1,4MW24AD001,ADITHYA S POOJARY,Data Science & GenAI - 3nd Yr- AI&DS,11,7,4,0,0.64,Critical,5 more,2026-10-07,0,Defaulter,Warning,Average,ADITHYA,AI & DS,3rd Year
2,4MW24AD002,ANKITHA PRASANNA NAYAK,Data Science & GenAI - 3nd Yr- AI&DS,11,11,0,0,1.00,Safe,✓ Met,2026-10-07,0,Safe,Excellent,Excellent,ANKITHA,AI & DS,3rd Year
3,4MW24AD003,ANUSHA KOTEGAR,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,0.91,Safe,✓ Met,2026-10-07,0,Safe,Excellent,Excellent,ANUSHA,AI & DS,3rd Year
4,4MW24AD004,ANVITA RAMESH NAIK,Data Science & GenAI - 3nd Yr- AI&DS,11,9,2,0,0.82,Warning,✓ Met,2026-10-07,0,Safe,Good,Good,ANVITA,AI & DS,3rd Year
5,4MW24AD005,ARPITHA M THANTRY,Data Science & GenAI - 3nd Yr- AI&DS,11,8,3,0,0.73,Critical,1 more,2026-10-07,0,Defaulter,Warning,Average,ARPITHA,AI & DS,3rd Year
6,4MW24AD007,ASNIYA,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,0.91,Safe,✓ Met,2026-10-07,0,Safe,Excellent,Excellent,ASNIYA,AI & DS,3rd Year
7,4MW24AD008,B. G PRARTHANA HEBBAR,Data Science & GenAI - 3nd Yr- AI&DS,11,10,1,0,0.91,Safe,✓ Met,2026-10-07,0,Safe,Excellent,Excellent,B.,AI & DS,3rd Year
8,4MW24AD009,BHARATHRAJ A NAYAK,Data Science & GenAI - 3nd Yr- AI&DS,11,4,7,0,0.36,Critical,17 more,2026-10-05,1,Defaulter,Critical,Poor,BHARATHRAJ,AI & DS,3rd Year
9,4MW24AD010,BHEEMA KATTIMANI,Data Science & GenAI - 3nd Yr- AI&DS,11,8,3,0,0.73,Critical,1 more,2026-10-07,0,Defaulter,Warning,Average,BHEEMA,AI & DS,3rd Year


In [24]:
"""Q20. Export the following to separate CSV files:
(a) All "Critical" students
(b) Summary statistics grouped by Status
(c) Top 10 students by Risk Score"""
# (a) Critical students
df[df['status'] == 'Critical'].to_csv('critical_students.csv', index=False)

# (b) Summary statistics grouped by Status
status_summary.to_csv('status_summary.csv')

# (c) Top 10 students by Risk Score
if 'risk_score' not in df.columns:
    df['risk_score'] = (df['absent'] * 0.4) + (df['consecutive_absences'] * 0.6)

df.nlargest(10, 'risk_score').to_csv('top_risk_students.csv', index=False)

print("All files exported successfully!")

All files exported successfully!
